# 🧭 Notebook 2: Vector Clocks — capturing causality

Lamport clocks give us a total order, but can't tell us whether two events are **causally related** or just happened to be numbered in sequence. A **vector clock** fixes that by keeping **one counter per process** instead of a single integer.

With 3 processes `A`, `B`, `C`, every event has a vector like `[A=2, B=5, C=1]`.

**Rules** (process `i`):
1. Local event → `V[i] += 1`.
2. Send → attach a copy of your vector to the message.
3. Receive `V'` → for every `j` set `V[j] = max(V[j], V'[j])`, then `V[i] += 1`.

**Comparison** of vectors `U` and `V`:
- `U ≤ V` if `U[i] ≤ V[i]` for **all** `i`.  → `U` **happened-before** `V`.
- `V ≤ U` symmetrically.
- Neither → **concurrent** (written `U ∥ V`).

Because every position records "what I know about that process," we never lose causal info.

## Learning objectives
- Implement a vector clock.
- Detect concurrent events that Lamport mis-ordered.
- Build a Dynamo-style shopping cart that **detects and resolves** conflicting writes.
- Know when real systems use (or avoid) vector clocks.


## 🔧 Setup

Same as Notebook 1: run `uv sync` once in the lab folder, then pick the `.venv` kernel in VS Code (top-right). Reload VS Code if it doesn't show up.

## 1. Vector-clock process

Below is a minimal `VCProcess`. Every vector is indexed **`[A, B, C]`** throughout this notebook — we'll print it that way so there's no confusion.

⚠️ Small but important: when sending, we send a **copy** of the vector. If we sent a reference, later mutations on the sender would silently rewrite the receiver's history — a classic distributed-systems-on-a-single-machine bug.

In [ ]:
def vle(u, v):
    """u ≤ v component-wise."""
    return all(a <= b for a, b in zip(u, v))

# The four outcomes are a closed set, so return a token rather than a sentence.
# (Matching on prose — `if result.startswith("u → v")` — is how a refactor of the
# wording silently turns a conflict-detector into a last-write-wins store.)
EQUAL, BEFORE, AFTER, CONCURRENT = "equal", "before", "after", "concurrent"

def compare(u, v):
    u, v = tuple(u), tuple(v)          # accept lists or tuples interchangeably
    if u == v:            return EQUAL         # same event / same knowledge
    if vle(u, v):         return BEFORE        # u happened before v
    if vle(v, u):         return AFTER         # v happened before u
    return CONCURRENT                          # neither dominates: u ∥ v

def describe(r):
    return {EQUAL: "equal",
            BEFORE: "u → v (u happened before v)",
            AFTER:  "v → u (v happened before u)",
            CONCURRENT: "concurrent (u ∥ v)"}[r]

def fmt(vc, names=("A", "B", "C")):
    return "[" + ", ".join(f"{n}={x}" for n, x in zip(names, vc)) + "]"


class VCProcess:
    def __init__(self, name, idx, n):
        self.name = name
        self.idx = idx
        self.vc = [0] * n
        self.history = []

    def local(self, label):
        self.vc[self.idx] += 1
        snap = tuple(self.vc)
        self.history.append((snap, f"{self.name}: {label}"))
        return snap

    def send(self, target, label):
        self.vc[self.idx] += 1
        snap = list(self.vc)             # COPY, not a reference
        self.history.append((tuple(snap), f"{self.name}: send '{label}' -> {target.name}"))
        return snap, label

    def recv(self, vc_in, label):
        self.vc = [max(a, b) for a, b in zip(self.vc, vc_in)]
        self.vc[self.idx] += 1
        snap = tuple(self.vc)
        self.history.append((snap, f"{self.name}: recv '{label}'"))
        return snap


A = VCProcess("A", 0, 3)
B = VCProcess("B", 1, 3)
C = VCProcess("C", 2, 3)

vc_a1 = A.local("write x=1")
m = A.send(B, "x=1");  B.recv(*m)
vc_b1 = B.local("write y=2")
m = B.send(C, "y=2");  C.recv(*m)
vc_c1 = C.local("write z=3")

# Independent events (no messages exchanged about them)
vc_a2 = A.local("indep A1")
vc_c2 = C.local("indep C1")

print(f"{'event':35s} vector")
print("-" * 65)
for vc, ev in A.history + B.history + C.history:
    print(f"{ev:35s} {fmt(vc)}")


### Compare the interesting pairs

Now we ask vector clocks the questions Lamport couldn't answer.

In [ ]:
pairs = [
    ("A: write x=1",  vc_a1, "C: write z=3",   vc_c1),  # causally linked via B
    ("A: indep A1",   vc_a2, "C: indep C1",    vc_c2),  # concurrent!
    ("A: indep A1",   vc_a2, "C: write z=3",   vc_c1),
]
for la, va, lb, vb in pairs:
    print(f"{la:20s} {fmt(va)}   vs   {lb:20s} {fmt(vb)}")
    print(f"   → {describe(compare(va, vb))}\n")

# The three answers, asserted.
assert compare(vc_a1, vc_c1) == BEFORE       # x=1 reached C through B
assert compare(vc_a2, vc_c2) == CONCURRENT   # <- the one Lamport got wrong
assert compare(vc_a2, vc_c1) == CONCURRENT
print("✔ the pair Lamport ordered is correctly reported as incomparable")


### Is it really a partial order?

"Concurrent" is only meaningful if the comparison genuinely refuses to order some pairs. A
subtly broken implementation — comparing sums, or comparing tuples lexicographically — still
returns sensible-looking answers for the examples above while quietly imposing a **total**
order, which is exactly the bug vector clocks exist to avoid.

So let's check the algebra directly, over a few thousand random clocks.

In [ ]:
import random as _random

rng = _random.Random(0)
clocks = [tuple(rng.randrange(0, 5) for _ in range(3)) for _ in range(400)]

concurrent_pairs = 0
for u in clocks:
    # Reflexive: every clock equals itself.
    assert compare(u, u) == EQUAL
    for v in clocks:
        r, r_rev = compare(u, v), compare(v, u)
        # Exactly one of the four outcomes, and it is consistent both ways round.
        assert {(EQUAL, EQUAL), (BEFORE, AFTER), (AFTER, BEFORE),
                (CONCURRENT, CONCURRENT)}.issuperset({(r, r_rev)}), (u, v, r, r_rev)
        # Antisymmetry: u → v and v → u cannot both hold unless they are equal.
        assert not (r == BEFORE and r_rev == BEFORE)
        if r == CONCURRENT:
            concurrent_pairs += 1

# THE property: a real partial order leaves many pairs unordered. A total order —
# the classic bug — would report zero concurrent pairs here.
total_pairs = len(clocks) ** 2
print(f"{concurrent_pairs:,} of {total_pairs:,} pairs are concurrent "
      f"({100 * concurrent_pairs / total_pairs:.0f}%)")
assert concurrent_pairs > 0, "no pair is concurrent — this is a TOTAL order, not a partial one"
assert concurrent_pairs > total_pairs * 0.3, concurrent_pairs

# Transitivity: u → v and v → w must imply u → w.
for _ in range(3000):
    u, v, w = rng.sample(clocks, 3)
    if compare(u, v) == BEFORE and compare(v, w) == BEFORE:
        assert compare(u, w) == BEFORE, (u, v, w)

# And the relationship to Lamport: happens-before implies a bigger scalar sum, but
# NOT the other way round. That gap is precisely the information Lamport throws away.
scalar_ordered_but_concurrent = [
    (u, v) for u in clocks for v in clocks
    if sum(u) < sum(v) and compare(u, v) == CONCURRENT
]
assert all(sum(u) < sum(v) for u in clocks for v in clocks if compare(u, v) == BEFORE)
assert scalar_ordered_but_concurrent, "expected scalar order to over-claim"
u, v = scalar_ordered_but_concurrent[0]
print(f"\nexample: {fmt(u)} has a smaller sum than {fmt(v)}, but they are concurrent")
print("✔ partial order verified: reflexive, antisymmetric, transitive, and genuinely partial")

🎯 The middle pair is the payoff. `[A=2, B=0, C=0]` and `[A=0, B=0, C=2]` are **incomparable** — neither is ≤ the other. Vector clocks correctly say: *concurrent*. Lamport couldn't.

> ⚠️ **Don't sort vector-clocked events onto a single line.** There's no total order. Drawing them linearly visually re-introduces the very lie we just removed.

## 2. 🛒 Real use: Amazon Dynamo's shopping cart

This is one of the most famous practical uses of vector clocks.

**Scenario.** Your cart is replicated on two servers `R1` and `R2`. You're on flaky WiFi. Two different "add item" requests land on the two replicas before they've synced with each other. What should happen?

- 👎 **Bad practice — last write wins (LWW):** whichever write gets the newer timestamp overwrites the other. You silently lose items from your cart. (This is how Cassandra resolves writes by default.)
- 👍 **Better — detect and merge:** vector clocks tell us the two writes are **concurrent**, so instead of picking one we merge them. For an *add-only* cart, the merge is the **union** of items.

> 🧠 **Important nuance:** vector clocks only **detect** conflicts. The **merge policy is application-defined**. Union works for our add-only cart. If users can also *remove* items, plain set-union is wrong (a re-added item looks the same as one that was never removed). Real Dynamo exposes the conflict to the client, or systems use richer data types (CRDTs — see §4).

In [ ]:
import copy

class CartReplica:
    """A replica that stores (items_set, vector_clock)."""
    def __init__(self, name, idx, n):
        self.name = name
        self.idx = idx
        self.vc = [0] * n
        self.items = set()

    def add(self, item):
        self.vc[self.idx] += 1
        self.items = self.items | {item}        # new set, no aliasing
        return copy.deepcopy((list(self.vc), set(self.items)))

    def snapshot(self):
        return (tuple(self.vc), frozenset(self.items))


# Two replicas of the same cart.
R1 = CartReplica("R1", 0, 2)
R2 = CartReplica("R2", 1, 2)

# User's phone adds "book" via R1. Their laptop adds "pen" via R2.
# The two replicas have NOT talked yet.
s1 = R1.add("book")
s2 = R2.add("pen")

print("R1 state:", fmt(R1.vc, ("R1", "R2")), "items =", R1.items)
print("R2 state:", fmt(R2.vc, ("R1", "R2")), "items =", R2.items)
print()
print("compare vector clocks:", compare(tuple(R1.vc), tuple(R2.vc)))


Both clocks are `[R1=1, R2=0]` vs `[R1=0, R2=1]` — **concurrent**. Now let's see both resolution strategies.

In [ ]:
# --- Bad practice: last-write-wins by wall-clock time ---
# Pretend R2's write had a slightly newer timestamp.
lww_winner = R2.items
print(f"❌ Last-write-wins  cart = {lww_winner}   (lost item from R1!)")

# --- Better: detect-and-merge using the vector clocks ---
def merge(v1, items1, v2, items2):
    cmp = compare(v1, v2)
    if cmp == BEFORE:
        return list(v2), set(items2)             # v2 dominates, keep it
    if cmp == AFTER:
        return list(v1), set(items1)             # v1 dominates, keep it
    # EQUAL or CONCURRENT: take the component-wise max of the clocks and union the
    # items. For an add-only cart, union is the right application-level merge —
    # see the note above about why it is NOT right once removes exist.
    merged_vc = [max(a, b) for a, b in zip(v1, v2)]
    merged_items = items1 | items2
    return merged_vc, merged_items

merged_vc, merged_items = merge(
    tuple(R1.vc), R1.items,
    tuple(R2.vc), R2.items,
)
print(f"✅ Vector-clock merge  vc = {fmt(merged_vc, ('R1','R2'))}  cart = {merged_items}")

# LWW silently dropped an item the user really added.
assert lww_winner == {"pen"} and "book" not in lww_winner
# The merge kept both, and the merged clock dominates BOTH inputs — so a later
# replica can tell this state is newer than either of the two it came from.
assert merged_items == {"book", "pen"}
assert compare(tuple(R1.vc), tuple(merged_vc)) == BEFORE
assert compare(tuple(R2.vc), tuple(merged_vc)) == BEFORE
print("✔ nothing lost, and the merged clock dominates both siblings")

# Merging is idempotent and commutative — both required, or replicas that gossip in
# different orders end up in different states.
again = merge(merged_vc, merged_items, merged_vc, merged_items)
assert (list(again[0]), again[1]) == (list(merged_vc), merged_items)
flipped = merge(tuple(R2.vc), R2.items, tuple(R1.vc), R1.items)
assert (list(flipped[0]), flipped[1]) == (list(merged_vc), merged_items)
print("✔ merge is idempotent and commutative")


### Where union stops working

The merge above is correct **for this cart**, and only because the cart is add-only. As soon
as a user can *remove* an item, set-union is wrong: a concurrent `remove("book")` and a replica
that still has `"book"` union back to `{"book"}`, so the removal is silently undone. The item
comes back from the dead — the bug is common enough to have a name.

Worth being precise about what failed: the vector clocks did their job perfectly. They detected
the conflict. It is the **merge policy** that is wrong, and no clock can fix that for you. This
is why real Dynamo hands siblings back to the application, and why CRDTs exist.

In [ ]:
# R1 removes "book"; R2 concurrently does something else. Both start from {book, pen}.
start_items = {"book", "pen"}
r1_vc, r1_items = [2, 1], start_items - {"book"}    # R1 removed book
r2_vc, r2_items = [1, 2], start_items | {"lamp"}    # R2 added lamp

assert compare(tuple(r1_vc), tuple(r2_vc)) == CONCURRENT   # detection works fine
_, naive = merge(tuple(r1_vc), r1_items, tuple(r2_vc), r2_items)
print("naive union merge ->", naive)
assert "book" in naive, "expected the removal to be undone"
print('💥 "book" is back. The delete was correctly detected as concurrent and then merged away.')

# A minimal CRDT fix: keep tombstones (an OR-Set style add/remove pair) so a removal
# is a fact to merge, not an absence to be overwritten by someone else's presence.
def crdt_merge(a_add, a_rem, b_add, b_rem):
    return (a_add | b_add), (a_rem | b_rem)

adds, removes = crdt_merge({"book", "pen"}, {"book"}, {"book", "pen", "lamp"}, set())
visible = adds - removes
print("\ntombstone merge   ->", visible)
assert visible == {"pen", "lamp"}
print('✔ the removal survives, because it is represented as data rather than as a gap')

## 3. 🌍 Where vector clocks show up in real systems

| System | What they use | Why |
|---|---|---|
| **Amazon Dynamo / Riak / Voldemort** | Vector clocks (or dotted version vectors) on every object | Detect concurrent writes, expose siblings to the client for merging. |
| **Cassandra** | Last-write-wins using synced wall-clock timestamps | Simpler ops; accepts silent data loss on conflict. Doesn't scale vector sizes. |
| **Git** | DAG of commits (each commit points to its parents) | Same *idea* as vector clocks: causal order via "who came before whom." Merge conflicts = concurrent commits. |
| **CRDTs (Redis, Automerge, Yjs, …)** | Vector clocks / dot contexts under the hood | Make merges automatic by restricting ops to ones that commute. |

### Lamport vs Vector — the trade

|  | Lamport | Vector clock |
|---|---|---|
| Size per event | 1 int | N ints (N = #participants) |
| Total order | yes | no |
| Causal order | partial | full |
| Detects concurrency? | **no** | **yes** |

### Why not always use vector clocks?

- **They grow with participants.** In a system where clients (not just servers) create vectors, the vector can balloon. Real systems **prune** old entries, or use **dotted version vectors** to bound size.
- **Membership changes are annoying.** Adding/removing a process means rewriting every vector.
- **Applications still need a merge policy.** Detecting a conflict is only half the story.

That's why many systems pick LWW (simpler, lossy) or CRDTs (harder to design, but merges are automatic).

## 🧠 Mini-exercises

Work these on paper first; they're quick.

1. Three processes with vectors ordered `[A, B, C]`. Is `[1, 2, 0]` concurrent with `[1, 1, 1]`? Why?
2. A fourth process `D` joins. What's the cleanest way to let existing vectors include it without breaking comparisons?
3. In the cart example, extend the scenario: `R1` adds `"book"`, syncs with `R2`, then `R2` adds `"pen"` and `R1` adds `"lamp"` concurrently. What are the final vector clocks, and does the merge still give the right cart?
4. Why would a `remove("book")` operation break the "just union the sets" merge? (This is why production systems use CRDTs or expose conflicts.)